# Baichuan-M2-32B QLoRA 微调（Notebook 版）

与 `train_qlora.py` 等价的 Notebook 版本，便于在服务器上交互式调试。

**前置条件**：已按 `Plan/20260514/Server_Operation_Manual.md` §3-§9 准备好环境、下载完 base 模型、切到 `ickg` 虚拟环境。

**与 .py 区别**：环境变量在第 1 个代码 cell 显式 `os.environ` 设置（必须在 import HF 库之前）；其余超参仍来自 `configs/train_config.yaml`。

In [ ]:
# ============================================================
# 必须最先执行：从 yaml 读 env 并写到 os.environ
# ============================================================
import os
from pathlib import Path
import yaml

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src' / 'Fine_tuning' / 'configs' / 'train_config.yaml').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('找不到 train_config.yaml，请在仓库根目录运行 notebook')
    REPO_ROOT = REPO_ROOT.parent

CONFIG_PATH = REPO_ROOT / 'src' / 'Fine_tuning' / 'configs' / 'train_config.yaml'
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

for k, v in (CFG.get('env') or {}).items():
    if v is not None:
        os.environ.setdefault(k, str(v))

print('HF_ENDPOINT =', os.environ.get('HF_ENDPOINT'))
print('HF_HOME     =', os.environ.get('HF_HOME'))
print('REPO_ROOT   =', REPO_ROOT)

In [ ]:
# ============================================================
# 导入 HF / Torch 相关库
# ============================================================
import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed,
)
from trl import SFTConfig, SFTTrainer

DTYPE = {'bfloat16': torch.bfloat16, 'float16': torch.float16, 'float32': torch.float32}
set_seed(CFG.get('seed', 42))

In [ ]:
# ============================================================
# 1. 4-bit 量化配置
# ============================================================
qcfg = CFG['quantization']
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=qcfg['load_in_4bit'],
    bnb_4bit_quant_type=qcfg['bnb_4bit_quant_type'],
    bnb_4bit_use_double_quant=qcfg['bnb_4bit_use_double_quant'],
    bnb_4bit_compute_dtype=DTYPE[qcfg['bnb_4bit_compute_dtype']],
)
print('[量化] ', qcfg)

In [ ]:
# ============================================================
# 2. tokenizer（包含训练用简化 chat template 切换）
# ============================================================
mcfg = CFG['model']
tokenizer = AutoTokenizer.from_pretrained(
    mcfg['name_or_path'],
    trust_remote_code=mcfg.get('trust_remote_code', True),
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 如果开启 assistant_only_loss，切换到训练专用简化模板（保存前会还原）
original_chat_template = None
if CFG.get('sft', {}).get('assistant_only_loss'):
    tpl_path = REPO_ROOT / 'src' / 'Fine_tuning' / 'training' / 'baichuan_m2_training_template.jinja'
    if not tpl_path.exists():
        raise FileNotFoundError(f'未找到训练 chat_template：{tpl_path}')
    original_chat_template = tokenizer.chat_template
    tokenizer.chat_template = tpl_path.read_text(encoding='utf-8')
    print(f'[ChatTemplate] 训练期间切换到简化模板：{tpl_path.name}')

In [ ]:
# ============================================================
# 3. 加载 4bit 模型并启用 k-bit 训练
# ============================================================
print(f'[模型] 加载 {mcfg["name_or_path"]} ...')

model_kwargs = {}
if mcfg.get('attn_implementation'):
    model_kwargs['attn_implementation'] = mcfg['attn_implementation']

model = AutoModelForCausalLM.from_pretrained(
    mcfg['name_or_path'],
    quantization_config=bnb_cfg,
    trust_remote_code=mcfg.get('trust_remote_code', True),
    dtype=DTYPE[mcfg.get('dtype') or mcfg['torch_dtype']],
    device_map='auto',
    **model_kwargs,
)
model.config.use_cache = False
if getattr(model, 'generation_config', None) is not None:
    model.generation_config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=CFG['training'].get('gradient_checkpointing', True),
)
print('[模型] 已加载并启用 k-bit 训练准备')

In [ ]:
# ============================================================
# 4. LoRA 配置
# ============================================================
lcfg = CFG['lora']
peft_config = LoraConfig(
    r=lcfg['r'],
    lora_alpha=lcfg['lora_alpha'],
    lora_dropout=lcfg['lora_dropout'],
    bias=lcfg['bias'],
    task_type=lcfg['task_type'],
    target_modules=lcfg['target_modules'],
)
print(f"[LoRA] r={lcfg['r']} alpha={lcfg['lora_alpha']} dropout={lcfg['lora_dropout']}")
print(f"        targets={lcfg['target_modules']}")

In [ ]:
# ============================================================
# 5. 数据集（messages 格式由 SFTTrainer 直接消化）
# ============================================================
sft_yaml = CFG['sft']
data_base = REPO_ROOT / sft_yaml['data_path']
train_path = data_base / sft_yaml['train_file']
val_path   = data_base / sft_yaml['val_file']
print('[数据] 训练集:', train_path)
print('[数据] 验证集:', val_path)

raw = load_dataset('json', data_files={'train': str(train_path), 'validation': str(val_path)})
print(raw)

# 烟测：只取前 N 条（调试用，正式训练时注释掉这两行）
# raw['train']      = raw['train'].select(range(50))
# raw['validation'] = raw['validation'].select(range(20))

In [ ]:
# ============================================================
# 6. SFTConfig（trl 0.16+ 用 max_length / assistant_only_loss）
# ============================================================
tcfg = CFG['training']
output_dir  = REPO_ROOT / tcfg['output_dir']
logging_dir = REPO_ROOT / tcfg.get('logging_dir', 'log/Fine_tuning/tensorboard')
output_dir.mkdir(parents=True, exist_ok=True)
logging_dir.mkdir(parents=True, exist_ok=True)

max_length_value = sft_yaml.get('max_length') or sft_yaml.get('max_seq_length')

sft_kwargs = dict(
    output_dir=str(output_dir),
    num_train_epochs=tcfg['num_train_epochs'],
    per_device_train_batch_size=tcfg['per_device_train_batch_size'],
    per_device_eval_batch_size=tcfg['per_device_eval_batch_size'],
    gradient_accumulation_steps=tcfg['gradient_accumulation_steps'],
    gradient_checkpointing=tcfg.get('gradient_checkpointing', True),
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=float(tcfg['learning_rate']),
    lr_scheduler_type=tcfg['lr_scheduler_type'],
    warmup_ratio=tcfg['warmup_ratio'],
    optim=tcfg['optim'],
    bf16=tcfg.get('bf16', True),
    max_grad_norm=tcfg['max_grad_norm'],
    weight_decay=tcfg['weight_decay'],
    logging_steps=tcfg['logging_steps'],
    logging_dir=str(logging_dir),
    save_steps=tcfg['save_steps'],
    save_total_limit=tcfg['save_total_limit'],
    eval_strategy=tcfg['eval_strategy'],
    eval_steps=tcfg['eval_steps'],
    load_best_model_at_end=tcfg['load_best_model_at_end'],
    metric_for_best_model=tcfg['metric_for_best_model'],
    greater_is_better=tcfg['greater_is_better'],
    report_to=tcfg['report_to'],
    remove_unused_columns=tcfg.get('remove_unused_columns', False),
    dataloader_num_workers=tcfg.get('dataloader_num_workers', 2),
    max_length=max_length_value,
    packing=sft_yaml.get('packing', False),
    seed=CFG.get('seed', 42),
)
if sft_yaml.get('assistant_only_loss'):
    sft_kwargs['assistant_only_loss'] = True

sft_config = SFTConfig(**sft_kwargs)
print('[SFTConfig] max_length=', max_length_value, ' assistant_only_loss=', sft_yaml.get('assistant_only_loss'))

In [ ]:
# ============================================================
# 7. SwanLab 初始化（可选；需提前 swanlab login）
# ============================================================
sw_cfg = CFG.get('swanlab') or {}
if 'swanlab' in tcfg.get('report_to', []) and sw_cfg.get('enabled'):
    import swanlab
    swanlab.init(
        project=sw_cfg.get('project', 'ICKG'),
        workspace=sw_cfg.get('workspace'),
        experiment_name=sw_cfg.get('experiment_name'),
        description=sw_cfg.get('description'),
        mode=sw_cfg.get('mode', 'cloud'),
        config={
            'model': mcfg['name_or_path'],
            'max_length': max_length_value,
            'lora_r': lcfg['r'],
            'lora_alpha': lcfg['lora_alpha'],
            'learning_rate': float(tcfg['learning_rate']),
            'num_train_epochs': tcfg['num_train_epochs'],
            'effective_batch_size': tcfg['per_device_train_batch_size'] * tcfg['gradient_accumulation_steps'],
            'load_in_4bit': qcfg['load_in_4bit'],
            'bnb_4bit_quant_type': qcfg['bnb_4bit_quant_type'],
            'optim': tcfg['optim'],
            'seed': CFG.get('seed', 42),
        },
    )
    print('[SwanLab] init OK')

In [ ]:
# ============================================================
# 8. 创建 SFTTrainer + 训练
# ============================================================
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=raw['train'],
    eval_dataset=raw['validation'],
    peft_config=peft_config,
    processing_class=tokenizer,
)
trainer.train()

In [ ]:
# ============================================================
# 9. 保存 LoRA adapter（保存前还原官方 chat template）
# ============================================================
adapter_dir = REPO_ROOT / CFG['adapter_dir']
adapter_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(adapter_dir))

if original_chat_template is not None:
    tokenizer.chat_template = original_chat_template
    print('[ChatTemplate] 已还原官方模板，将随 tokenizer 一并写入 adapter/')
tokenizer.save_pretrained(str(adapter_dir))
print('[完成] LoRA adapter + tokenizer →', adapter_dir)

try:
    import swanlab
    if swanlab.get_run() is not None:
        swanlab.finish()
except Exception:
    pass